[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brilliantbeaver/alexpose/blob/main/penny/gavd3/09_mask_geometry_ablation.ipynb)

# 09. Mask geometry ablation under matched compute

Notebook 04 pretrained one S-JEPA model with uniform masking over ten neurologic landmarks. This notebook retrains the identical recipe under four masking geometries with equal masked-token budgets, equal seeds, and equal epoch counts, then compares downstream classifiers, collapse diagnostics, and a minimal cadence probe.

**Research use only.** This tutorial does not diagnose a person or validate a clinical device.

**Run it:** locally, use `uv sync` then `uv run jupyter lab` from this folder. In Colab, use the badge and run the setup cell. Restart the kernel after changing `penny/gavd3/.env`.

**Keep the walk visible:** notebook 01 opens the source video, notebook 02 shows frame, bbox, and skeleton alignment, and notebook 03 proves which joints may become prediction targets. Revisit those views whenever a mask or classifier result looks surprising.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/brilliantbeaver/alexpose.git"

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy", "pandas", "scipy", "scikit-learn", "matplotlib",
        "seaborn", "torch", "tqdm", "python-dotenv", "yt-dlp[default]",
        "opencv-python-headless", "mediapipe<1", "joblib", "pyarrow",
    ])
    clone_dir = Path("/content/alexpose")
    if not (clone_dir / ".git").exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)])
    os.chdir(clone_dir)


def find_project_root(start=None):
    env_root = os.getenv("ALEXPOSE_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / ".git").exists() and (candidate / "data" / "gavd").exists():
            return candidate
        print(f"Ignoring invalid ALEXPOSE_ROOT: {candidate}")
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() and (candidate / "data" / "gavd").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
TUTORIAL_DIR = PROJECT_ROOT / "penny" / "gavd3"

try:
    from dotenv import load_dotenv
    load_dotenv(TUTORIAL_DIR / ".env", override=False)
    load_dotenv(PROJECT_ROOT / ".env", override=False)
except Exception:
    pass

MODE = os.getenv("GAVD3_MODE", "smoke").strip().lower()
if MODE not in {"smoke", "real"}:
    raise ValueError("GAVD3_MODE must be smoke or real")
if MODE == "smoke":
    print(
        "SMOKE MODE: hand-authored motions test code paths only. "
        "They have no pathophysiological or clinical validity."
    )

PREFERRED_ROOT = Path(
    os.getenv(
        "GAVD4_ROOT",
        "/Users/pmui/vaults/worldmodels/gait/skeleton-jepa/gavd4",
    )
).expanduser()

requested_data = os.getenv("GAVD4_DATA_DIR") or os.getenv("GAVD_DATA_GAVD_DIR")
if requested_data and Path(requested_data).expanduser().exists():
    DATA_GAVD_DIR = Path(requested_data).expanduser()
elif requested_data:
    print(f"Ignoring missing GAVD CSV path: {Path(requested_data).expanduser()}")
    if (PREFERRED_ROOT / "data-gavd").exists():
        DATA_GAVD_DIR = PREFERRED_ROOT / "data-gavd"
    else:
        DATA_GAVD_DIR = PROJECT_ROOT / "data" / "gavd"
elif (PREFERRED_ROOT / "data-gavd").exists():
    DATA_GAVD_DIR = PREFERRED_ROOT / "data-gavd"
else:
    DATA_GAVD_DIR = PROJECT_ROOT / "data" / "gavd"

requested_youtube = os.getenv("GAVD4_YOUTUBE_DIR") or os.getenv("GAVD_YOUTUBE_DIR")
if requested_youtube:
    YOUTUBE_DIR = Path(requested_youtube).expanduser()
elif PREFERRED_ROOT.exists():
    YOUTUBE_DIR = PREFERRED_ROOT / "youtube"
else:
    YOUTUBE_DIR = PROJECT_ROOT / "penny" / "gavd3" / "work" / "youtube"

CACHE_DIR = Path(
    os.getenv("GAVD3_CACHE_DIR", TUTORIAL_DIR / "work" / "cache")
).expanduser()
ARTIFACT_ROOT = Path(
    os.getenv("GAVD3_ARTIFACT_DIR", TUTORIAL_DIR / "work" / "artifacts")
).expanduser()
ARTIFACT_DIR = ARTIFACT_ROOT / MODE
POSE_DIR = ARTIFACT_DIR / "poses"

for folder in [CACHE_DIR, ARTIFACT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR / "xdg-cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
from IPython.display import SVG, display


def show_tutorial_svg(filename):
    '''Render a repository SVG reliably in local Jupyter and Colab.'''
    path = TUTORIAL_DIR / "images" / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Missing tutorial figure {path}. Clone the full alexpose repository."
        )
    display(SVG(filename=str(path)))

print(f"mode: {MODE}")
print(f"project: {PROJECT_ROOT}")
print(f"GAVD CSVs: {DATA_GAVD_DIR}")
print(f"YouTube cache: {YOUTUBE_DIR}")
print(f"artifacts: {ARTIFACT_DIR}")

## Why the mask is a hypothesis

Notebook 03 derived ten neurologic landmarks (the `MASK_KEYPOINTS` set) from high-priority feature mappings, and notebook 04 pretrained with the `uniform_neurologic_mask` sampler. Every batch hides about 60 percent of the eligible tokens inside those ten joints, which is only about 18 percent of all joint-time tokens. Nothing in those notebooks proves that this geometry is optimal.

The mask decides three things at once: which tokens the view encoder never sees, which tokens the predictor must reconstruct, and how those two groups change over time. Each of those choices can change what the representation learns. The literature mapping tells you a sensible set of joints to study, not whether hiding only those joints helps the model.

A fair ablation needs matched compute so that any score difference comes from geometry alone:

- Identical model configuration, optimizer, learning-rate schedule, and EMA schedule.
- The same pretraining corpus (the normal sequences only) and the same seeds.
- The same number of epochs and optimizer updates for every arm.
- The same number of masked tokens per sequence in every arm, so the view encoder keeps the same number of tokens at every step.
- One knob changed at a time relative to the neurologic-10 baseline.

Runtime guards use `os.getenv` only for documented variables, and the notebook has two modes:

|Setting|Smoke default|Real default|
|---:|---:|---:|
|GAVD3_MODE|smoke|real|
|Epochs per arm|12|300|
|Seeds per mask|0, 1, 2|42|
|Frames per clip|32|64|
|Masked fraction of neurologic-eligible tokens|0.60|0.60|

Smoke mode uses hand-authored synthetic motions that exercise every code path. Their numbers are labelled smoke fixtures and have no clinical meaning. Real mode requires the cached pose corpus from notebook 02 and fails loudly when it is missing.

In [ ]:
BLAZEPOSE_33 = [
    "NOSE", "LEFT_EYE_INNER", "LEFT_EYE", "LEFT_EYE_OUTER",
    "RIGHT_EYE_INNER", "RIGHT_EYE", "RIGHT_EYE_OUTER", "LEFT_EAR",
    "RIGHT_EAR", "MOUTH_LEFT", "MOUTH_RIGHT", "LEFT_SHOULDER",
    "RIGHT_SHOULDER", "LEFT_ELBOW", "RIGHT_ELBOW", "LEFT_WRIST",
    "RIGHT_WRIST", "LEFT_PINKY", "RIGHT_PINKY", "LEFT_INDEX",
    "RIGHT_INDEX", "LEFT_THUMB", "RIGHT_THUMB", "LEFT_HIP",
    "RIGHT_HIP", "LEFT_KNEE", "RIGHT_KNEE", "LEFT_ANKLE",
    "RIGHT_ANKLE", "LEFT_HEEL", "RIGHT_HEEL", "LEFT_FOOT_INDEX",
    "RIGHT_FOOT_INDEX",
]
MASK_KEYPOINTS = [11, 12, 23, 24, 25, 26, 27, 28, 31, 32]
assert [BLAZEPOSE_33[i] for i in MASK_KEYPOINTS] == [
    "LEFT_SHOULDER", "RIGHT_SHOULDER", "LEFT_HIP", "RIGHT_HIP",
    "LEFT_KNEE", "RIGHT_KNEE", "LEFT_ANKLE", "RIGHT_ANKLE",
    "LEFT_FOOT_INDEX", "RIGHT_FOOT_INDEX",
]



CONDITIONS = ["normal", "parkinsons", "stroke", "cerebralpalsy", "myopathic"]

In [ ]:
def synthetic_gait_sequence(condition="normal", frames=64, seed=0):
    '''Create a code-path fixture, not a physiological disease simulation.'''
    rng = np.random.default_rng(seed)
    phase = np.linspace(0.0, 4.0 * np.pi, frames, endpoint=False)
    seq = np.zeros((frames, 33, 4), dtype=np.float32)
    seq[..., 3] = 1.0
    base = {
        11: (0.42, 0.28), 12: (0.58, 0.28),
        23: (0.45, 0.52), 24: (0.55, 0.52),
        25: (0.44, 0.70), 26: (0.56, 0.70),
        27: (0.43, 0.89), 28: (0.57, 0.89),
        29: (0.42, 0.92), 30: (0.58, 0.92),
        31: (0.39, 0.94), 32: (0.61, 0.94),
    }
    for joint, (x, y) in base.items():
        seq[:, joint, 0] = x
        seq[:, joint, 1] = y
    amplitude = 0.045
    lift = 0.025
    if condition == "parkinsons":
        amplitude *= 0.45
        lift *= 0.45
    if condition == "myopathic":
        seq[:, [11, 12], 0] += 0.03 * np.sin(phase)[:, None]
        seq[:, [23, 24], 0] += 0.018 * np.sin(phase)[:, None]
    for joint, knee, foot, offset in [(27, 25, 31, 0.0), (28, 26, 32, np.pi)]:
        wave = np.sin(phase + offset)
        if condition == "stroke" and joint == 27:
            wave = 0.35 * wave
        if condition == "cerebralpalsy":
            seq[:, knee, 1] -= 0.045
            seq[:, joint, 1] -= 0.02
        seq[:, joint, 0] += amplitude * wave
        seq[:, knee, 0] += 0.4 * amplitude * wave
        seq[:, foot, 0] += amplitude * wave
        seq[:, joint, 1] -= lift * np.maximum(wave, 0.0)
        seq[:, foot, 1] -= 0.7 * lift * np.maximum(wave, 0.0)
    seq[..., :3] += rng.normal(0.0, 0.0025, seq[..., :3].shape)
    return seq


def synthetic_corpus(conditions=None, n_per_condition=10, frames=64, seed=42):
    if conditions is None:
        conditions = [
            "normal", "parkinsons", "stroke", "cerebralpalsy", "myopathic"
        ]
    records = []
    counter = 0
    for condition in conditions:
        for sample in range(n_per_condition):
            records.append({
                "condition": condition,
                "sequence_id": f"smoke_{condition}_{sample:03d}",
                "video_id": f"smoke_video_{condition}_{sample // 2:02d}",
                "sequence": synthetic_gait_sequence(
                    condition=condition,
                    frames=frames,
                    seed=seed + counter,
                ),
            })
            counter += 1
    return records

In [ ]:
def interpolate_low_visibility(sequence, threshold=0.45, max_gap=4):
    '''Fill only short internal gaps and preserve the original validity mask.

    Long gaps and sequence ends are never extrapolated. Their coordinates remain
    missing until center_and_scale converts them to an explicit zero sentinel.
    They can never become S-JEPA prediction targets.
    '''
    sequence = np.asarray(sequence, dtype=np.float32).copy()
    if sequence.ndim != 3 or sequence.shape[1:] != (33, 4):
        raise ValueError(f"Expected [T, 33, 4], received {sequence.shape}")
    visibility = np.nan_to_num(sequence[..., 3], nan=0.0)
    finite = np.isfinite(sequence[..., :3]).all(axis=-1)
    valid = (visibility >= threshold) & finite
    filled = valid.copy()
    for joint in range(33):
        observed = np.flatnonzero(valid[:, joint])
        for left, right in zip(observed[:-1], observed[1:]):
            gap = int(right - left - 1)
            if not 0 < gap <= max_gap:
                continue
            fraction = (
                np.arange(1, gap + 1, dtype=np.float32) / (gap + 1)
            )[:, None]
            sequence[left + 1:right, joint, :3] = (
                sequence[left, joint, :3][None, :] * (1.0 - fraction)
                + sequence[right, joint, :3][None, :] * fraction
            )
            filled[left + 1:right, joint] = True
        sequence[~filled[:, joint], joint, :3] = np.nan
    sequence[..., 3] = visibility
    return sequence, valid


def center_and_scale(sequence, eps=1e-6):
    sequence = np.asarray(sequence, dtype=np.float32).copy()
    xyz = sequence[..., :3]
    left_hip, right_hip = xyz[:, 23], xyz[:, 24]
    left_ok = np.isfinite(left_hip).all(axis=1)
    right_ok = np.isfinite(right_hip).all(axis=1)
    pelvis = np.full((len(xyz), 3), np.nan, dtype=np.float32)
    pelvis[left_ok & right_ok] = 0.5 * (
        left_hip[left_ok & right_ok] + right_hip[left_ok & right_ok]
    )
    pelvis[left_ok & ~right_ok] = left_hip[left_ok & ~right_ok]
    pelvis[right_ok & ~left_ok] = right_hip[right_ok & ~left_ok]
    pelvis_ok = np.isfinite(pelvis).all(axis=1)
    fallback = np.median(pelvis[pelvis_ok], axis=0) if pelvis_ok.any() else np.zeros(3)
    pelvis[~np.isfinite(pelvis).all(axis=1)] = fallback
    xyz = xyz - pelvis[:, None, :]
    shoulder_width = np.linalg.norm(xyz[:, 11, :2] - xyz[:, 12, :2], axis=-1)
    hip_width = np.linalg.norm(xyz[:, 23, :2] - xyz[:, 24, :2], axis=-1)
    body_scale = np.nanmedian(np.maximum(shoulder_width, hip_width))
    if not np.isfinite(body_scale) or body_scale < eps:
        body_scale = 1.0
    sequence[..., :3] = np.nan_to_num(
        xyz / body_scale, nan=0.0, posinf=0.0, neginf=0.0
    )
    return np.nan_to_num(sequence, nan=0.0, posinf=0.0, neginf=0.0)


def temporal_resize(array, frames):
    array = np.asarray(array)
    if len(array) == frames:
        return array.copy()
    if len(array) < 2:
        return np.repeat(array, frames, axis=0)
    old_t = np.linspace(0.0, 1.0, len(array))
    new_t = np.linspace(0.0, 1.0, frames)
    flat = array.reshape(len(array), -1)
    resized = np.stack(
        [np.interp(new_t, old_t, flat[:, i]) for i in range(flat.shape[1])],
        axis=1,
    )
    return resized.reshape(frames, *array.shape[1:]).astype(array.dtype)


def prepare_sequence(
    sequence,
    frames=64,
    visibility_threshold=0.45,
    max_gap=4,
):
    cleaned, valid = interpolate_low_visibility(
        sequence, visibility_threshold, max_gap=max_gap
    )
    cleaned = center_and_scale(cleaned)
    cleaned = temporal_resize(cleaned, frames)
    valid = temporal_resize(valid.astype(np.float32), frames) >= 0.5
    return cleaned[..., :3].astype(np.float32), valid.astype(bool)

In [ ]:
def uniform_neurologic_mask(valid_patch, mask_fraction=0.60, seed=None):
    """Sample eligible joint-time tokens uniformly, without motion scores.

    valid_patch has shape [B, S, V]. True means that a patch can be a target.
    The returned mask has the same shape. True means hidden from the view encoder.
    """
    valid_patch = np.asarray(valid_patch, dtype=bool)
    if valid_patch.ndim != 3 or valid_patch.shape[2] != 33:
        raise ValueError(f"Expected [B, S, 33], received {valid_patch.shape}")
    if not 0.0 < mask_fraction < 1.0:
        raise ValueError("mask_fraction must be between 0 and 1")
    rng = np.random.default_rng(seed)
    eligible_joint = np.zeros(33, dtype=bool)
    eligible_joint[MASK_KEYPOINTS] = True
    eligible = valid_patch & eligible_joint[None, None, :]
    counts = eligible.reshape(len(eligible), -1).sum(axis=1)
    if np.any(counts < 2):
        raise ValueError("Each sample needs at least two valid eligible tokens")
    n_mask = max(1, int(np.floor(counts.min() * mask_fraction)))
    n_mask = min(n_mask, int(counts.min()) - 1)
    mask = np.zeros_like(eligible)
    for batch_index in range(len(mask)):
        candidates = np.flatnonzero(eligible[batch_index].reshape(-1))
        chosen = rng.choice(candidates, size=n_mask, replace=False)
        mask[batch_index].reshape(-1)[chosen] = True
    forbidden = sorted(set(range(33)).difference(MASK_KEYPOINTS))
    assert not mask[:, :, forbidden].any()
    assert mask.reshape(len(mask), -1).any(axis=1).all()
    assert (~mask).reshape(len(mask), -1).any(axis=1).all()
    return mask


def mask_audit(mask, valid_patch):
    mask = np.asarray(mask, dtype=bool)
    valid_patch = np.asarray(valid_patch, dtype=bool)
    eligible_joint = np.zeros(33, dtype=bool)
    eligible_joint[MASK_KEYPOINTS] = True
    eligible = valid_patch & eligible_joint[None, None, :]
    masked_counts = mask.reshape(len(mask), -1).sum(axis=1)
    eligible_counts = eligible.reshape(len(mask), -1).sum(axis=1)
    per_sample_ratio = masked_counts / eligible_counts
    touched = np.flatnonzero(mask.any(axis=(0, 1))).tolist()
    return {
        "masked_keypoints": touched,
        "masked_names": [BLAZEPOSE_33[i] for i in touched],
        "global_fraction": float(mask.mean()),
        "eligible_mask_fraction_min": float(per_sample_ratio.min()),
        "eligible_mask_fraction_mean": float(per_sample_ratio.mean()),
        "eligible_mask_fraction_max": float(per_sample_ratio.max()),
        "forbidden_count": int(mask[:, :, sorted(set(range(33)) - set(MASK_KEYPOINTS))].sum()),
    }

In [ ]:
import copy
import math
import torch
from torch import nn


class SkeletonPatchEncoder(nn.Module):
    def __init__(
        self,
        frames=64,
        joints=33,
        coordinate_dim=3,
        segment_length=4,
        embed_dim=64,
        depth=2,
        heads=4,
        dropout=0.0,
    ):
        super().__init__()
        if frames % segment_length:
            raise ValueError("frames must be divisible by segment_length")
        self.frames = frames
        self.joints = joints
        self.coordinate_dim = coordinate_dim
        self.segment_length = segment_length
        self.segments = frames // segment_length
        self.embed_dim = embed_dim
        self.patch_embed = nn.Linear(segment_length * coordinate_dim, embed_dim)
        self.time_pos = nn.Parameter(torch.randn(self.segments, embed_dim) * 0.02)
        self.joint_pos = nn.Parameter(torch.randn(joints, embed_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)

    def patchify(self, x):
        batch, frames, joints, channels = x.shape
        expected = (self.frames, self.joints, self.coordinate_dim)
        if (frames, joints, channels) != expected:
            raise ValueError(f"Expected [B, {expected}], received {x.shape}")
        patches = x.reshape(
            batch, self.segments, self.segment_length, joints, channels
        )
        patches = patches.permute(0, 1, 3, 2, 4).contiguous()
        return patches.flatten(3)

    def positioned_tokens(self, x):
        tokens = self.patch_embed(self.patchify(x))
        return (
            tokens
            + self.time_pos[None, :, None, :]
            + self.joint_pos[None, None, :, :]
        )

    def forward(self, x, keep_mask=None):
        tokens = self.positioned_tokens(x)
        batch = len(tokens)
        flat = tokens.reshape(batch, self.segments * self.joints, self.embed_dim)
        if keep_mask is not None:
            keep_mask = keep_mask.reshape(batch, -1)
            kept_per_sample = keep_mask.sum(dim=1)
            if not torch.equal(kept_per_sample, kept_per_sample[:1].expand_as(kept_per_sample)):
                raise ValueError("Each sample must keep the same number of tokens")
            flat = flat[keep_mask].reshape(batch, int(kept_per_sample[0]), self.embed_dim)
        return self.norm(self.blocks(flat))


class SkeletonPredictor(nn.Module):
    def __init__(
        self,
        segments,
        joints,
        encoder_dim=64,
        predictor_dim=64,
        depth=2,
        heads=4,
        dropout=0.0,
    ):
        super().__init__()
        self.segments = segments
        self.joints = joints
        self.encoder_to_predictor = nn.Linear(encoder_dim, predictor_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_dim))
        nn.init.normal_(self.mask_token, std=0.02)
        self.time_pos = nn.Parameter(torch.randn(segments, predictor_dim) * 0.02)
        self.joint_pos = nn.Parameter(torch.randn(joints, predictor_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=predictor_dim,
            nhead=heads,
            dim_feedforward=predictor_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(predictor_dim)
        self.output = nn.Linear(predictor_dim, encoder_dim)

    def forward(self, visible_features, target_mask):
        batch = len(visible_features)
        target_mask = target_mask.reshape(batch, self.segments * self.joints)
        visible_mask = ~target_mask
        visible = self.encoder_to_predictor(visible_features)
        full = self.mask_token.expand(
            batch, self.segments * self.joints, -1
        ).clone()
        full[visible_mask] = visible.reshape(-1, visible.shape[-1])
        positions = (
            self.time_pos[:, None, :] + self.joint_pos[None, :, :]
        ).reshape(1, self.segments * self.joints, -1)
        full = full + positions
        predicted = self.output(self.norm(self.blocks(full)))
        return predicted[target_mask].reshape(batch, -1, predicted.shape[-1])


class SJEPAGait(nn.Module):
    def __init__(
        self,
        frames=64,
        joints=33,
        coordinate_dim=3,
        segment_length=4,
        embed_dim=64,
        encoder_depth=2,
        predictor_depth=2,
        heads=4,
    ):
        super().__init__()
        self.view_encoder = SkeletonPatchEncoder(
            frames, joints, coordinate_dim, segment_length,
            embed_dim, encoder_depth, heads,
        )
        self.target_encoder = copy.deepcopy(self.view_encoder)
        for parameter in self.target_encoder.parameters():
            parameter.requires_grad_(False)
        self.predictor = SkeletonPredictor(
            self.view_encoder.segments,
            joints,
            embed_dim,
            embed_dim,
            predictor_depth,
            heads,
        )
        self.register_buffer("target_center", torch.zeros(embed_dim))

    def forward(self, view, target, target_mask):
        visible_features = self.view_encoder(view, keep_mask=~target_mask)
        predicted = self.predictor(visible_features, target_mask)
        with torch.no_grad():
            all_targets = self.target_encoder(target)
            flat_mask = target_mask.reshape(len(target), -1)
            selected = all_targets[flat_mask].reshape(
                len(target), -1, all_targets.shape[-1]
            )
        return predicted, selected

    @torch.no_grad()
    def update_target(self, momentum):
        for target_parameter, view_parameter in zip(
            self.target_encoder.parameters(), self.view_encoder.parameters()
        ):
            target_parameter.mul_(momentum).add_(
                view_parameter, alpha=1.0 - momentum
            )

    @torch.no_grad()
    def update_center(self, targets, beta=0.9):
        batch_center = targets.mean(dim=(0, 1))
        self.target_center.mul_(beta).add_(batch_center, alpha=1.0 - beta)


def sjepa_cross_entropy(
    predicted,
    targets,
    center,
    predictor_temperature=0.10,
    target_temperature=0.06,
):
    target_prob = torch.softmax(
        (targets - center[None, None, :]) / target_temperature,
        dim=-1,
    ).detach()
    prediction_log_prob = torch.log_softmax(
        predicted / predictor_temperature,
        dim=-1,
    )
    return -(target_prob * prediction_log_prob).sum(dim=-1).mean()


def cosine_ema(step, total_steps, start=0.996, end=1.0):
    progress = min(max(step / max(total_steps - 1, 1), 0.0), 1.0)
    return end - (end - start) * (math.cos(math.pi * progress) + 1.0) / 2.0


LEFT_RIGHT_PAIRS = [
    (1, 4), (2, 5), (3, 6), (7, 8), (9, 10), (11, 12),
    (13, 14), (15, 16), (17, 18), (19, 20), (21, 22),
    (23, 24), (25, 26), (27, 28), (29, 30), (31, 32),
]


def geometric_view(
    x,
    max_degrees=8.0,
    translate=0.03,
    flip_probability=0.0,
):
    """Apply one sequence-wide transform per sample.

    Rotation is around the relative vertical y axis, so x and z are mixed.
    Flip defaults to off because laterality can matter for stroke. If enabled,
    coordinates are reflected and every left-right landmark pair is swapped.
    """
    view = x.clone()
    present = view.abs().sum(dim=-1) > 1e-8
    batch = len(view)
    angles = (
        torch.rand(batch, device=x.device) * 2.0 - 1.0
    ) * math.radians(max_degrees)
    cosine, sine = torch.cos(angles), torch.sin(angles)
    original_x = view[..., 0].clone()
    original_z = view[..., 2].clone()
    rotated_x = cosine[:, None, None] * original_x + sine[:, None, None] * original_z
    rotated_z = -sine[:, None, None] * original_x + cosine[:, None, None] * original_z
    view[..., 0] = rotated_x
    view[..., 2] = rotated_z
    offsets = (torch.rand(batch, 1, 1, 2, device=x.device) * 2.0 - 1.0) * translate
    view[..., :2] += offsets
    if flip_probability > 0:
        flip = torch.rand(batch, device=x.device) < flip_probability
        for batch_index in torch.where(flip)[0].tolist():
            view[batch_index, ..., 0] *= -1.0
            original = view[batch_index].clone()
            original_present = present[batch_index].clone()
            for left, right in LEFT_RIGHT_PAIRS:
                view[batch_index, :, left] = original[:, right]
                view[batch_index, :, right] = original[:, left]
                present[batch_index, :, left] = original_present[:, right]
                present[batch_index, :, right] = original_present[:, left]
    view = view.masked_fill(~present[..., None], 0.0)
    return view

In [ ]:
def pose_records_from_cache(pose_dir=POSE_DIR, conditions=CONDITIONS):
    records = []
    for condition in conditions:
        folder = Path(pose_dir) / condition
        for path in sorted(folder.glob("*.npz")):
            data = np.load(path, allow_pickle=False)
            required = {
                "sequence", "sequence_id", "video_id", "condition",
                "frame_numbers", "crop_bounds", "fps", "source_csv",
                "source_video", "pose_model", "pose_model_sha256",
                "extraction_version",
            }
            missing = required.difference(data.files)
            if missing:
                raise ValueError(
                    f"Stale pose cache {path} is missing {sorted(missing)}. "
                    "Re-extract it with notebook 02."
                )
            sequence = data["sequence"].astype(np.float32)
            if sequence.ndim != 3 or sequence.shape[1:] != (33, 4):
                raise ValueError(f"Bad pose shape in {path}: {sequence.shape}")
            stored_condition = str(data["condition"].item())
            if stored_condition != condition:
                raise ValueError(
                    f"Pose condition {stored_condition} does not match folder {condition}"
                )
            if len(data["frame_numbers"]) != len(sequence):
                raise ValueError(f"Frame and pose lengths differ in {path}")
            records.append({
                "condition": condition,
                "sequence_id": str(data["sequence_id"].item()),
                "video_id": str(data["video_id"].item()),
                "source_video": str(data["source_video"].item()),
                "fps": float(data["fps"].item()),
                "extraction_version": str(data["extraction_version"].item()),
                "pose_model_sha256": str(data["pose_model_sha256"].item()),
                "sequence": sequence,
                "path": str(path),
            })
    return records


def load_records_for_mode(conditions=CONDITIONS, smoke_per_condition=10, frames=64):
    if MODE == "smoke":
        records = synthetic_corpus(
            conditions=conditions,
            n_per_condition=smoke_per_condition,
            frames=frames,
        )
        print(f"Explicit smoke corpus: {len(records)} synthetic sequences")
        return records
    records = pose_records_from_cache(conditions=conditions)
    counts = pd.Series([r["condition"] for r in records]).value_counts()
    missing = [condition for condition in conditions if counts.get(condition, 0) == 0]
    if missing:
        raise FileNotFoundError(
            f"Real mode requires cached pose sequences for {missing}. "
            "Run notebook 02 first."
        )
    print(f"Real pose corpus: {len(records)} sequences")
    return records

## Load the two corpora

You need two views of the same records. The self-supervised pretraining corpus is the normal subset: that is what every mask arm is trained on, mirroring notebook 04. The evaluation corpus is the full five-condition set, because the classifier probes need all conditions to score mask arms on downstream tasks.

The preparation pipeline is identical to notebook 04: interpolate only short internal visibility gaps, center on the mid-hip, scale by shoulder or hip width, resize to a fixed frame count, and drop to x, y, relative z plus the validity mask. Invalid tokens can never become targets.

Real mode validates the full census (96 sequences from 18 source videos) exactly like notebook 06. Missing poses raise an instructive `FileNotFoundError` that tells you to rerun notebook 02 with `GAVD_EXTRACT_POSES=1`, `GAVD_EXTRACT_CONDITIONS=all`, and `GAVD_MAX_SEQUENCES=0`. Every normal sequence must also clear the neurologic coverage threshold before pretraining starts.

In [ ]:
FRAMES = int(os.getenv("SJEPA_FRAMES", "32" if MODE == "smoke" else "64"))
SEGMENT_LENGTH = 4
if FRAMES % SEGMENT_LENGTH:
    raise ValueError("SJEPA_FRAMES must be divisible by 4")
SEGMENTS = FRAMES // SEGMENT_LENGTH

records_all = load_records_for_mode(
    conditions=CONDITIONS,
    smoke_per_condition=10,
    frames=FRAMES,
)
prepared_all = [
    prepare_sequence(record["sequence"], frames=FRAMES)
    for record in records_all
]
all_xyz = np.stack([item[0] for item in prepared_all]).astype(np.float32)
all_valid = np.stack([item[1] for item in prepared_all]).astype(bool)
sequence_ids = np.asarray([record["sequence_id"] for record in records_all])
video_ids = np.asarray([record["video_id"] for record in records_all])
labels = np.asarray([record["condition"] for record in records_all])
if len(set(sequence_ids.tolist())) != len(sequence_ids):
    raise ValueError("Cached pose sequence IDs must be unique")

if MODE == "real":
    expected_sequences = {
        "cerebralpalsy": 16, "myopathic": 47, "normal": 12,
        "parkinsons": 9, "stroke": 12,
    }
    expected_videos = {
        "cerebralpalsy": 2, "myopathic": 10, "normal": 1,
        "parkinsons": 2, "stroke": 3,
    }
    actual_sequences = pd.Series(labels).value_counts().to_dict()
    actual_videos = (
        pd.DataFrame({"condition": labels, "video_id": video_ids})
        .groupby("condition")["video_id"]
        .nunique()
        .to_dict()
    )
    if actual_sequences != expected_sequences:
        raise FileNotFoundError(
            f"Real mode needs all 96 cached poses but found {actual_sequences}. "
            "Run notebook 02 with GAVD_EXTRACT_POSES=1, "
            "GAVD_EXTRACT_CONDITIONS=all, and GAVD_MAX_SEQUENCES=0 first."
        )
    if actual_videos != expected_videos:
        raise ValueError(f"Unexpected 18-video census: {actual_videos}")

normal_rows = labels == "normal"
normal_xyz = all_xyz[normal_rows]
normal_valid = all_valid[normal_rows]
normal_ids = sequence_ids[normal_rows]
normal_video_ids = video_ids[normal_rows]
expected_normal = int(os.getenv("GAVD_EXPECTED_NORMAL_SEQUENCES", "12"))
if MODE == "real" and len(normal_ids) != expected_normal:
    raise ValueError(
        f"Expected {expected_normal} normal pose files, found {len(normal_ids)}. "
        "In penny/gavd3/.env set GAVD_EXTRACT_POSES=1, "
        "GAVD_EXTRACT_CONDITIONS=normal, and GAVD_MAX_SEQUENCES=0; "
        "restart the kernel and rerun notebook 02 through its extraction cell."
    )
min_coverage = float(os.getenv("GAVD_MIN_NEURO_COVERAGE", "0.50"))
coverage_report = pd.DataFrame({
    "sequence_id": normal_ids,
    "video_id": normal_video_ids,
    "neurologic_observed_fraction": normal_valid[:, :, MASK_KEYPOINTS].mean(axis=(1, 2)),
})
display(coverage_report)
below = coverage_report[
    coverage_report["neurologic_observed_fraction"] < min_coverage
]
if not below.empty:
    raise ValueError(
        f"{len(below)} normal sequences fall below the neurologic coverage "
        f"threshold {min_coverage:.2f}. Review extraction first."
    )
print("evaluation corpus:", all_xyz.shape, "from",
      len(set(video_ids.tolist())), "videos")
print("pretraining corpus (normal only):", normal_xyz.shape, "from",
      len(set(normal_video_ids.tolist())), "video(s)")
if MODE == "real" and len(set(normal_video_ids.tolist())) == 1:
    print(
        "Warning: every normal sequence shares one source video. The ablation "
        "is transductive with respect to that video, exactly like notebook 04."
    )

## Four samplers, one change at a time

Every sampler receives the same `valid_patch` matrix of shape [B, S, 33], where True means the joint-time token may be a target. Every sampler returns a mask of the same shape, where True means the token is hidden from the view encoder and becomes a prediction target. All four draw the same per-sequence budget, called n*, equal to what the neurologic-10 baseline would mask: floor of 0.60 times the smallest per-sample count of valid neurologic tokens, batch-common.

|Arm|Eligible pool|Sampling weights|Change vs baseline|
|---|---|---|---|
|neurologic-10|10 neurologic joints|uniform|none, the baseline itself|
|random-10|all 33 joints|uniform|drop the joint restriction|
|motion-aware-10|10 neurologic joints|proportional to per-token displacement|add the motion prior|
|full-body-33|all 33 joints|uniform, stratified over segments|match the budget over the whole body|

One naming note. The suffix 10 records that the arm spends the masked-token budget derived from the ten neurologic joints at 60 percent, and 33 records the full 33-joint grid. random-10 draws n* tokens from the entire grid exchangeably, so by chance some temporal segments lose several tokens while others lose none. full-body-33 splits the same n* budget as evenly as possible across temporal segments and draws uniformly inside each segment, so every segment receives equal mask pressure. The pair (random-10, full-body-33) therefore isolates temporal stratification, and the pair (neurologic-10, full-body-33) isolates the joint restriction at equal budget.

Motion weighting uses the prepared xyz only. The motion score of a patch is the Euclidean distance between its first and last frame for that joint, so a fast-moving ankle scores higher than a quiet shoulder. Invalid cells get a zero weight and can never be chosen. An epsilon floor keeps every eligible cell drawable, and when all motion is zero the arm falls back to uniform.

All four samplers carry the same two assertions: invalid tokens are never masked, and the masked count is identical for every sample in a batch. A dedicated test cell re-checks those invariants together with the equal-budget property across arms, and saves a per-arm audit table plus a geometry figure.

In [ ]:
MASK_FRACTION = 0.60  # same eligible-token fraction as notebook 04 pretraining


def ablation_mask_budget(valid_patch, mask_fraction=0.60):
    """Return the batch-common masked-token budget of the neurologic-10 arm.

    This replicates the count rule inside uniform_neurologic_mask so that every
    ablation arm hides exactly the same number of tokens per sequence.
    """
    valid_patch = np.asarray(valid_patch, dtype=bool)
    eligible_joint = np.zeros(33, dtype=bool)
    eligible_joint[MASK_KEYPOINTS] = True
    eligible = valid_patch & eligible_joint[None, None, :]
    counts = eligible.reshape(len(eligible), -1).sum(axis=1)
    if np.any(counts < 2):
        raise ValueError("Each sample needs at least two valid neurologic tokens")
    n_mask = max(1, int(np.floor(counts.min() * mask_fraction)))
    return min(n_mask, int(counts.min()) - 1)


def assert_mask_invariants(mask, valid_patch, n_mask):
    """Shared guard: never mask invalid tokens, keep counts equal per batch."""
    mask = np.asarray(mask, dtype=bool)
    valid_patch = np.asarray(valid_patch, dtype=bool)
    if mask.shape != valid_patch.shape:
        raise ValueError(f"mask {mask.shape} must match valid {valid_patch.shape}")
    masked_invalid = mask & (~valid_patch)
    assert not masked_invalid.any(), "A sampler masked an invalid token"
    counts = mask.reshape(len(mask), -1).sum(axis=1)
    assert np.all(counts == counts[0]), "Masked counts must match across a batch"
    assert int(counts[0]) == int(n_mask), (
        f"Expected {n_mask} masked tokens per sample, got {counts[0]}"
    )
    assert mask.reshape(len(mask), -1).any(axis=1).all(), "No token was masked"
    assert (~mask).reshape(len(mask), -1).any(axis=1).all(), (
        "The view encoder must keep at least one token per sample"
    )


def neurologic10_mask(valid_patch, xyz, mask_fraction=0.60, seed=None):
    """Baseline arm: verbatim uniform sampler restricted to the neurologic set."""
    mask = uniform_neurologic_mask(
        valid_patch, mask_fraction=mask_fraction, seed=seed
    )
    assert_mask_invariants(mask, valid_patch, ablation_mask_budget(
        valid_patch, mask_fraction
    ))
    forbidden = sorted(set(range(33)).difference(MASK_KEYPOINTS))
    assert not mask[:, :, forbidden].any(), "neurologic-10 touched a forbidden joint"
    return mask


def random_body_mask(valid_patch, xyz, mask_fraction=0.60, seed=None):
    """random-10: uniform over every valid token of all 33 joints.

    It spends the neurologic-10 budget (same masked count per sequence) but
    drops the joint restriction. The draw is exchangeable over the whole grid,
    so temporal segments are not balanced on purpose.
    """
    valid_patch = np.asarray(valid_patch, dtype=bool)
    n_mask = ablation_mask_budget(valid_patch, mask_fraction)
    rng = np.random.default_rng(seed)
    mask = np.zeros_like(valid_patch)
    for batch_index in range(len(mask)):
        candidates = np.flatnonzero(valid_patch[batch_index].reshape(-1))
        if len(candidates) < n_mask:
            raise ValueError(
                f"Sample {batch_index} has only {len(candidates)} valid tokens; "
                f"cannot draw the matched budget of {n_mask}."
            )
        chosen = rng.choice(candidates, size=n_mask, replace=False)
        mask[batch_index].reshape(-1)[chosen] = True
    assert_mask_invariants(mask, valid_patch, n_mask)
    return mask


def per_token_displacement(xyz, valid_patch, segment_length=4):
    """L2 distance between the first and last frame of each patch token.

    Inputs are the prepared coordinates, so invalid frames carry the zero
    sentinel; their tokens are multiplied out to zero displacement here and are
    excluded from candidates upstream by the validity matrix.
    """
    xyz = np.asarray(xyz, dtype=np.float32)
    valid_patch = np.asarray(valid_patch, dtype=bool)
    batch, frames, joints, channels = xyz.shape
    if channels != 3 or joints != 33:
        raise ValueError(f"Expected [B, T, 33, 3], received {xyz.shape}")
    if frames % segment_length:
        raise ValueError("frames must be divisible by segment_length")
    segments = frames // segment_length
    if valid_patch.shape != (batch, segments, joints):
        raise ValueError(
            f"valid_patch {valid_patch.shape} does not match [B, {segments}, 33]"
        )
    reshaped = xyz.reshape(batch, segments, segment_length, joints, channels)
    magnitude = np.linalg.norm(
        reshaped[:, :, -1] - reshaped[:, :, 0], axis=-1
    )
    magnitude = magnitude * valid_patch.astype(np.float32)
    return magnitude


def motion_aware_neurologic_mask(valid_patch, xyz, mask_fraction=0.60, seed=None):
    """motion-aware-10: neurologic set, probability proportional to motion.

    Each eligible token receives weight displacement + epsilon, so a token is
    never impossible to draw even when its joint is momentarily still. The
    sampled count still equals the neurologic-10 budget.
    """
    valid_patch = np.asarray(valid_patch, dtype=bool)
    eligible_joint = np.zeros(33, dtype=bool)
    eligible_joint[MASK_KEYPOINTS] = True
    eligible = valid_patch & eligible_joint[None, None, :]
    n_mask = ablation_mask_budget(valid_patch, mask_fraction)
    rng = np.random.default_rng(seed)
    weights = per_token_displacement(xyz, valid_patch) + 1e-6
    weights = weights * eligible
    mask = np.zeros_like(valid_patch)
    for batch_index in range(len(mask)):
        candidates = np.flatnonzero(eligible[batch_index].reshape(-1))
        if len(candidates) < n_mask:
            raise ValueError(
                f"Sample {batch_index} has only {len(candidates)} valid "
                f"neurologic tokens; cannot draw the matched budget of {n_mask}."
            )
        probabilities = weights[batch_index].reshape(-1)[candidates]
        probabilities = probabilities / probabilities.sum()
        chosen = rng.choice(candidates, size=n_mask, replace=False, p=probabilities)
        mask[batch_index].reshape(-1)[chosen] = True
    assert_mask_invariants(mask, valid_patch, n_mask)
    forbidden = sorted(set(range(33)).difference(MASK_KEYPOINTS))
    assert not mask[:, :, forbidden].any(), (
        "motion-aware-10 touched a forbidden joint"
    )
    return mask


def full_body_stratified_mask(valid_patch, xyz, mask_fraction=0.60, seed=None):
    """full-body-33: all 33 joints, budget split evenly across time segments.

    The neurologic-10 budget is divided as evenly as possible over the S
    temporal segments. Each segment then draws uniformly among its valid
    joints. If a very sparse segment cannot meet its quota, the deficit is
    borrowed from the remaining valid tokens elsewhere, so the total masked
    count stays exactly equal to the neurologic-10 budget.
    """
    valid_patch = np.asarray(valid_patch, dtype=bool)
    batch, segments, joints = valid_patch.shape
    n_mask = ablation_mask_budget(valid_patch, mask_fraction)
    quota, remainder = divmod(n_mask, segments)
    quota_per_segment = np.full(segments, quota, dtype=int)
    quota_per_segment[:remainder] += 1
    rng = np.random.default_rng(seed)
    mask = np.zeros_like(valid_patch)
    for batch_index in range(len(mask)):
        left_over = quota_per_segment.copy()
        for segment in range(segments):
            if left_over[segment] <= 0:
                continue
            candidates = np.flatnonzero(valid_patch[batch_index, segment])
            if len(candidates) >= left_over[segment]:
                chosen = rng.choice(
                    candidates, size=int(left_over[segment]), replace=False
                )
                mask[batch_index, segment, chosen] = True
                left_over[segment] = 0
            else:
                mask[batch_index, segment, candidates] = True
                left_over[segment] -= len(candidates)
        deficit = int(left_over.sum())
        if deficit > 0:
            open_cells = np.flatnonzero(
                (valid_patch[batch_index] & ~mask[batch_index]).reshape(-1)
            )
            if len(open_cells) < deficit:
                raise ValueError(
                    f"Sample {batch_index} is too sparse to hold the matched "
                    f"budget of {n_mask} full-body tokens."
                )
            chosen = rng.choice(open_cells, size=deficit, replace=False)
            mask[batch_index].reshape(-1)[chosen] = True
    assert_mask_invariants(mask, valid_patch, n_mask)
    return mask


SAMPLER_NAMES = ["neurologic-10", "random-10", "motion-aware-10", "full-body-33"]
SAMPLERS = {
    "neurologic-10": neurologic10_mask,
    "random-10": random_body_mask,
    "motion-aware-10": motion_aware_neurologic_mask,
    "full-body-33": full_body_stratified_mask,
}

In [ ]:
import matplotlib.pyplot as plt

test_count = min(4, len(normal_xyz))
test_xyz = np.ascontiguousarray(normal_xyz[:test_count])
test_valid = np.asarray(normal_valid[:test_count], dtype=bool)
batch, frames, joints, channels = test_xyz.shape
test_segments = frames // SEGMENT_LENGTH
valid_patch = (
    test_valid.reshape(batch, test_segments, SEGMENT_LENGTH, joints)
    .all(axis=2)
)
expected_budget = ablation_mask_budget(valid_patch, MASK_FRACTION)
test_masks = {}
audit_rows = []
for name in SAMPLER_NAMES:
    mask = SAMPLERS[name](valid_patch, test_xyz, MASK_FRACTION, seed=7)
    test_masks[name] = mask
    counts = mask.reshape(batch, -1).sum(axis=1)
    assert np.all(counts == expected_budget), name
    forbidden = sorted(set(range(33)).difference(MASK_KEYPOINTS))
    touched_joints = sorted(set(np.flatnonzero(mask.any(axis=(0, 1))).tolist()))
    audit_rows.append({
        "sampler": name,
        "masked_tokens_per_sequence": int(counts[0]),
        "global_fraction": float(mask.mean()),
        "forbidden_count": int(mask[:, :, forbidden].sum()),
        "touched_joints": len(touched_joints),
        "touched_names": ";".join(BLAZEPOSE_33[j] for j in touched_joints),
        "mean_masked_displacement": float(
            per_token_displacement(test_xyz, valid_patch)[mask].mean()
        ),
    })
audit_table = pd.DataFrame(audit_rows)
display(audit_table)
audit_table.to_csv(ARTIFACT_DIR / "09_mask_sampler_audit.csv", index=False)

fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True, sharey=True)
for axis, name in zip(axes.ravel(), SAMPLER_NAMES):
    axis.imshow(test_masks[name][0].T, aspect="auto", cmap="Blues")
    axis.set_title(f"{name}: {int(test_masks[name][0].sum())} masked tokens")
    axis.set_xlabel("temporal segment")
    axis.set_ylabel("BlazePose joint index")
    for joint in MASK_KEYPOINTS:
        axis.axhline(joint - 0.5, color="tab:red", linewidth=0.4, alpha=0.5)
fig.suptitle("One exemplar mask per arm (red rows mark the neurologic set)")
fig.tight_layout()
fig.savefig(
    ARTIFACT_DIR / "09_mask_geometry_samplers.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()
print("sampler invariants passed for every arm; budget per sequence:",
      expected_budget)

## A shared matched-compute training function

One function, `train_arm`, trains every arm. It reproduces the notebook 04 loop: AdamW with weight decay, warmup then cosine learning-rate decay, gradient clipping, an EMA target encoder updated with a cosine momentum schedule toward 1, a running target center, and per-batch geometric views (small rotation, translation, no flip).

Masks are redrawn at every batch, exactly like notebook 04. The random stream is seeded as `MASK_SEED_BASE + seed * 1_000_003 + step`, so the same seed gives every arm the same underlying random sequence and the arms differ only by their sampling structure.

Runtime controls:

|Environment variable|Smoke default|Real default|Purpose|
|---|---|---|---|
|SJEPA_ABLATION_EPOCHS|12|300 (quick profile: 20)|epochs per arm|
|SJEPA_ABLATION_SEEDS|0,1,2|42|seeds, one per arm set|
|SJEPA_RUN_PROFILE|smoke|recommended|real quick vs recommended|
|SJEPA_FRAMES|32|64|clip frames (notebook 04 convention)|

Real mode trains four arms on 12 normal sequences for 300 epochs each, which is about 900 optimizer updates per arm. The real quick profile exists only to validate the real-data code path, and the notebook prints that warning. Training saves the full per-epoch history to `09_ablation_training_history.csv` and one checkpoint per arm, `sjepa_ablation_<mask>_seed<seed>.pt`, for auditability.

In [ ]:
if MODE == "smoke":
    RUN_PROFILE = "smoke"
    EMBED_DIM, ENCODER_DEPTH, PREDICTOR_DEPTH, HEADS = 32, 1, 1, 4
    DEFAULT_EPOCHS = "12"
    DEFAULT_SEEDS = "0,1,2"
    EMA_START = 0.996
    LEARNING_RATE = 3e-4
else:
    RUN_PROFILE = os.getenv("SJEPA_RUN_PROFILE", "recommended").strip().lower()
    if RUN_PROFILE not in {"quick", "recommended"}:
        raise ValueError("SJEPA_RUN_PROFILE must be quick or recommended")
    EMBED_DIM, ENCODER_DEPTH, PREDICTOR_DEPTH, HEADS = 96, 4, 2, 4
    DEFAULT_EPOCHS = "20" if RUN_PROFILE == "quick" else "300"
    DEFAULT_SEEDS = "42"
    EMA_START = 0.996 if RUN_PROFILE == "quick" else 0.999
    LEARNING_RATE = 1e-3

EPOCHS = int(os.getenv("SJEPA_ABLATION_EPOCHS", DEFAULT_EPOCHS))
SEED_TEXT = os.getenv("SJEPA_ABLATION_SEEDS", DEFAULT_SEEDS)
SEEDS = sorted({int(item) for item in SEED_TEXT.split(",") if item.strip()})
if not SEEDS:
    raise ValueError("SJEPA_ABLATION_SEEDS must list at least one seed")
if EPOCHS < 1:
    raise ValueError("SJEPA_ABLATION_EPOCHS must be at least 1")
BATCH_SIZE = 4
config = {
    "frames": FRAMES,
    "joints": 33,
    "coordinate_dim": 3,
    "segment_length": SEGMENT_LENGTH,
    "embed_dim": EMBED_DIM,
    "encoder_depth": ENCODER_DEPTH,
    "predictor_depth": PREDICTOR_DEPTH,
    "heads": HEADS,
}
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("mode:", MODE)
print("profile:", RUN_PROFILE)
print("epochs per arm:", EPOCHS)
print("seeds:", SEEDS)
print("device:", device)
print("config:", config)
print("total training runs:", len(SEEDS) * len(SAMPLER_NAMES))
if MODE == "real" and RUN_PROFILE == "quick":
    print("QUICK PROFILE: validate the real-data path only; do not report it.")

In [ ]:
import torch.nn.functional as F


@torch.no_grad()
def collapse_diagnostics(model, arrays, batch_size=8):
    """Mean feature std and mean off-diagonal pair cosine of pooled tokens."""
    model.target_encoder.eval()
    pooled = []
    for start in range(0, len(arrays), batch_size):
        batch_tensor = torch.tensor(
            arrays[start:start + batch_size],
            dtype=torch.float32,
            device=device,
        )
        tokens = model.target_encoder(batch_tensor)
        pooled.append(tokens.mean(dim=1).cpu())
    pooled = torch.cat(pooled, dim=0)
    feature_std = pooled.std(dim=0, unbiased=False).mean().item()
    normalized = F.normalize(pooled, dim=1)
    similarities = normalized @ normalized.T
    if len(pooled) > 1:
        off_diagonal = similarities[
            ~torch.eye(len(pooled), dtype=torch.bool)
        ].mean().item()
    else:
        off_diagonal = float("nan")
    return feature_std, off_diagonal

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

MASK_SEED_BASE = 100003


def train_arm(mask_name, seed, epochs):
    """Train one mask arm with the shared matched-compute loop.

    Only the mask sampler differs between calls. Seeding is identical for the
    same seed, so arms differ by masking structure and nothing else.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = SJEPAGait(**config).to(device)
    trainable = [parameter for parameter in model.parameters()
                 if parameter.requires_grad]
    assert not any(parameter.requires_grad
                   for parameter in model.target_encoder.parameters())
    optimizer = torch.optim.AdamW(
        trainable,
        lr=LEARNING_RATE,
        betas=(0.9, 0.95),
        weight_decay=0.05,
    )
    dataset = TensorDataset(
        torch.tensor(normal_xyz, dtype=torch.float32),
        torch.tensor(normal_valid, dtype=torch.bool),
    )
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        dataset,
        batch_size=min(BATCH_SIZE, len(dataset)),
        shuffle=True,
        generator=generator,
        drop_last=False,
    )
    total_steps = epochs * len(loader)
    warmup_steps = max(1, min(len(loader), total_steps // 10))

    def learning_rate_factor(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(total_steps - warmup_steps - 1, 1)
        return 0.5 + 0.5 * (1.0 + math.cos(math.pi * progress)) / 2.0

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=learning_rate_factor
    )
    history = []
    global_step = 0
    for epoch in range(epochs):
        model.train()
        batch_rows = []
        for coordinates, valid in loader:
            coordinates = coordinates.to(device)
            valid = valid.to(device)
            valid_patch = (
                valid.reshape(len(valid), SEGMENTS, SEGMENT_LENGTH, 33)
                .all(dim=2)
                .cpu()
                .numpy()
            )
            xyz_batch = coordinates.detach().cpu().numpy()
            mask_seed = MASK_SEED_BASE + seed * 1_000_003 + global_step
            mask_np = SAMPLERS[mask_name](
                valid_patch, xyz_batch, MASK_FRACTION, seed=mask_seed
            )
            target_mask = torch.tensor(mask_np, device=device)
            view = geometric_view(
                coordinates,
                max_degrees=8.0,
                translate=0.03,
                flip_probability=0.0,
            )
            prediction, target = model(view, coordinates, target_mask)
            loss = sjepa_cross_entropy(prediction, target, model.target_center)
            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite loss for {mask_name} seed {seed} "
                    f"step {global_step}"
                )
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
            optimizer.step()
            scheduler.step()
            momentum = cosine_ema(
                global_step, total_steps, start=EMA_START, end=1.0
            )
            model.update_target(momentum)
            model.update_center(target, beta=0.9)
            batch_rows.append({
                "loss": float(loss.detach().cpu()),
                "global_mask_fraction": float(mask_np.mean()),
                "ema_momentum": momentum,
            })
            global_step += 1
        feature_std, mean_cosine = collapse_diagnostics(model, normal_xyz)
        summary = pd.DataFrame(batch_rows).mean(numeric_only=True).to_dict()
        summary.update({
            "epoch": epoch + 1,
            "feature_std": feature_std,
            "mean_pair_cosine": mean_cosine,
            "learning_rate": optimizer.param_groups[0]["lr"],
        })
        history.append(summary)
        show_every = max(1, epochs // 8)
        if (
            epoch + 1 <= 3
            or (epoch + 1) % show_every == 0
            or epoch + 1 == epochs
        ):
            print(
                f"arm {mask_name:16s} seed {seed:02d}  "
                f"epoch {epoch + 1:03d}  loss {summary['loss']:.4f}  "
                f"std {feature_std:.3f}  cos {mean_cosine:.3f}"
            )
    return model, pd.DataFrame(history)

In [ ]:
arm_models = {}
history_frames = []
for seed in SEEDS:
    for name in SAMPLER_NAMES:
        print(f"\n=== train {name} seed {seed} ===")
        model, history_df = train_arm(name, seed, EPOCHS)
        history_df.insert(0, "mode", MODE)
        history_df.insert(0, "seed", seed)
        history_df.insert(0, "mask", name)
        history_frames.append(history_df)
        model = model.to("cpu")
        model.eval()
        arm_models[(name, seed)] = model
        checkpoint_path = ARTIFACT_DIR / (
            f"sjepa_ablation_{name.replace(' ', '_')}_seed{seed}.pt"
        )
        torch.save({
            "mode": MODE,
            "mask_name": name,
            "seed": seed,
            "config": config,
            "epochs": EPOCHS,
            "model_state": model.state_dict(),
            "final_loss": float(history_df["loss"].iloc[-1]),
            "final_feature_std": float(history_df["feature_std"].iloc[-1]),
        }, checkpoint_path)
        print("saved:", checkpoint_path.name)

all_history = pd.concat(history_frames, ignore_index=True)
all_history.to_csv(
    ARTIFACT_DIR / "09_ablation_training_history.csv", index=False
)
print("training history rows:", len(all_history))
last_epoch = all_history[all_history["epoch"] == EPOCHS]
display(
    last_epoch.groupby(["mask", "seed"], as_index=False)[
        ["loss", "feature_std", "mean_pair_cosine"]
    ].mean()
)

model = arm_models[(SAMPLER_NAMES[0], SEEDS[0])]
print("reference arm for the pooled-embedding demo:",
      SAMPLER_NAMES[0], "seed", SEEDS[0])

## Evaluation protocol

The pooled-embedding helper from notebooks 05 and 06 runs verbatim below. Its last lines execute on the neurologic-10 reference arm to demonstrate the 384-dimensional vector, and a loop after it pools every other arm.

Three measurements decide the comparison:

(i) Five-class RandomForest on the pooled embeddings. Lane 1 reproduces the exp5-style sequence split, which in real mode is the exact legacy 47/21 permutation over the 68 exp5 sequence IDs, labelled video-confounded because it is a comparability lane. Smoke mode uses a clearly labelled stratified stand-in because synthetic IDs cannot reproduce exp5. Lane 2 is a video-grouped cross-validation over source videos. It requires at least two source videos in every class; real mode has one normal video, so it prints a BLOCKED diagnostic exactly as notebook 06 did instead of inventing a number.

(ii) Collapse diagnostics. Every epoch records the mean feature standard deviation and the mean off-diagonal pair cosine of pooled latent vectors on the normal corpus. Falling variance or rising similarity would signal representation collapse, not a healthy mask.

(iii) A minimal cadence probe so this notebook stays independent of notebook 08. The proxy is the number of ankle horizontal-x cycles per frame, estimated from the prepared xyz with `scipy` peak finding. A RidgeCV regressor with five-fold cross-validation reports R2 from the latent vectors, alongside two baselines: pooled statistics of the raw prepared coordinates, and detector-missingness features. Smoke sequences share one synthetic cadence, so smoke adds a tiny deterministic jitter keyed to each sequence ID purely to give the Ridge target nonzero variance; it is labelled as a fixture.

Every result table is written to ARTIFACT_DIR with a 09_ prefix.

In [ ]:
def masked_mean_std(tokens, mask):
    weights = torch.as_tensor(
        mask, dtype=tokens.dtype, device=tokens.device
    ).unsqueeze(-1)
    denominator = weights.sum(dim=1).clamp_min(1.0)
    mean = (tokens * weights).sum(dim=1) / denominator
    variance = (
        (tokens - mean[:, None, :]).square() * weights
    ).sum(dim=1) / denominator
    return mean, variance.clamp_min(0.0).sqrt()


@torch.no_grad()
def pooled_embeddings(model, arrays, validity, batch_size=8):
    vectors = []
    segments = model.target_encoder.segments
    segment_length = model.target_encoder.segment_length
    dimension = model.target_encoder.embed_dim
    for start in range(0, len(arrays), batch_size):
        batch = torch.tensor(
            arrays[start:start + batch_size],
            dtype=torch.float32,
        )
        tokens = model.target_encoder(batch).reshape(
            len(batch), segments, 33, dimension
        )
        valid_patch = np.asarray(
            validity[start:start + batch_size], dtype=bool
        ).reshape(
            len(batch), segments, segment_length, 33
        ).all(axis=2)
        global_tokens = tokens.reshape(len(batch), -1, dimension)
        neuro_tokens = tokens[:, :, MASK_KEYPOINTS].reshape(
            len(batch), -1, dimension
        )
        global_mean, global_std = masked_mean_std(
            global_tokens, valid_patch.reshape(len(batch), -1)
        )
        neuro_mean, neuro_std = masked_mean_std(
            neuro_tokens,
            valid_patch[:, :, MASK_KEYPOINTS].reshape(len(batch), -1),
        )
        vector = torch.cat(
            [
                global_mean,
                global_std,
                neuro_mean,
                neuro_std,
            ],
            dim=1,
        )
        vectors.append(vector.cpu())
    return torch.cat(vectors).numpy()

In [ ]:
embeddings_by_arm = {}
for name in SAMPLER_NAMES:
    for seed in SEEDS:
        arm_model = arm_models[(name, seed)]
        arm_embeddings = pooled_embeddings(arm_model, all_xyz, all_valid)
        assert arm_embeddings.shape[0] == len(sequence_ids)
        assert np.isfinite(arm_embeddings).all()
        embeddings_by_arm[(name, seed)] = arm_embeddings
        print(f"embeddings {name:16s} seed {seed:02d}: "
              f"{arm_embeddings.shape}")
print("pooled embeddings ready for every arm")

In [ ]:
EXP5_ORDER = [
    "cljvvsucg00043n6l4evgn7q4", "cljr5jk0h000n3n6la34mkdfz",
    "cljo8eumx00683n6le2s7myd4", "cljr5fc5d000b3n6lkvc71zyl",
    "cljr5iki0000j3n6lwi8z5nh6", "cljo8cyv500603n6lyl148tmg",
    "cljo8g74m006g3n6l6kuxy9cf", "cljo8hcfv006k3n6lgrx0fcpx",
    "cljo8fdke006c3n6lr8bzjjgi", "cljo8c0sw005w3n6l9ulr2eg2",
    "cljo8e32t00643n6l37ncjeic", "cljr5hwxc000f3n6lof5w9tyt",
    "cljarhldg00d13n6l7utw0lqn", "cljo83yl800513n6lfglf5jn8",
    "cljawdoej000d3n6ll5ysj34f", "cljarmcm700dh3n6lxw24hxgx",
    "cljaxabfg003a3n6l95mrlcry", "cljarj6rf00d53n6ljtivf4q7",
    "cljarlbch00dd3n6l9jaubrxq", "cljarp08600dt3n6lxa7sysiv",
    "cljarpts600dx3n6l98zqf6yn", "cljaroguw00dp3n6lhfddiadu",
    "cljargosy00cx3n6lm6atozrz", "cljarn9oy00dl3n6l8pg9exfg",
    "cljarqtuz00e13n6lrnox6mfs", "cljaxbq22003i3n6lmzka93uq",
    "cljarkax200d93n6lukx52g1t", "cljax9d2p00363n6lx043s1m7",
    "cljaxb5y2003e3n6lj1j9qvav", "cljo84jju00553n6lmgm4dqtb",
    "cljawd01m00093n6l4kx9020l", "cljawb5nf00043n6l38chxsod",
    "cljnz7jnj000w3n6lku3pfvtm", "cljnz6vg1000s3n6lxxknbe72",
    "cljnz3l34000c3n6ldapq560j", "cljnz5sb1000o3n6lntosswwz",
    "cljan9b4p00043n6ligceanyp", "cljanb45y00083n6lmh1qhydd",
    "cljnyzwbo00043n6lugyldlhu", "cljnz4y5a000k3n6lkv4b1rjn",
    "cljnz4e8u000g3n6l1luikppo", "cljo340cm002a3n6low42ugvh",
    "cljo32xnz00223n6lvxzyif3y", "cljo33m8400263n6l4xxsl6ku",
    "cljo2y1f7001e3n6lt1wgacw6", "cljo3b2dy002l3n6l270vpzp8",
    "cljo2yqzp001i3n6lg75p7wtq", "cljo39ok9002h3n6ldr0w5sey",
    "cljo30lnz001q3n6lopfty7q5", "cljo2wwu7001a3n6ljmqm39l6",
    "cljo2zn41001m3n6lhbvww48i", "cljo32ik2001y3n6lmmnu0sgo",
    "cljo32213001u3n6lel97up5f", "cljar9bqo00c43n6l2u5zmlru",
    "cljarbn1y00cg3n6l1u4i0d5l", "cljardvzg00cs3n6loetskba6",
    "cljas71p600fv3n6lk1rzl7y5", "cljas134500f73n6lkfbjfayp",
    "cljarcfa700ck3n6lfww83ig1", "cljar9t8o00c83n6ltculhoct",
    "cljas1yfs00fb3n6lna38ui6i", "cljarar9t00cc3n6lqhi9udoc",
    "cljas5esv00fn3n6lewd5xqdl", "cljas4dqj00fj3n6ldw6wiwpy",
    "cljar878f00c03n6ly2v2ay88", "cljas04fw00f33n6lm5cvx9g6",
    "cljas2sou00ff3n6lasppj8h2", "cljarcy3g00co3n6lzsn1x034",
]


def exp5_exact_split(order, train_portion=0.70):
    """Reproduce the original notebook's legacy NumPy permutation."""
    np.random.seed(42)
    permutation = np.random.permutation(len(order))
    split_index = int(train_portion * len(order))
    train_ids = [order[index] for index in permutation[:split_index]]
    test_ids = [order[index] for index in permutation[split_index:]]
    return train_ids, test_ids


assert len(EXP5_ORDER) == 68

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

CLASS_ORDER = [
    "cerebralpalsy", "myopathic", "normal", "parkinsons", "stroke"
]


def make_rf():
    return Pipeline([
        ("scale", StandardScaler()),
        ("rf", RandomForestClassifier(
            n_estimators=100,
            max_depth=5,
            max_features="sqrt",
            bootstrap=True,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )),
    ])


def score_predictions(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


id_to_index = {
    sequence_id: index for index, sequence_id in enumerate(sequence_ids)
}
if MODE == "real":
    missing = [item for item in EXP5_ORDER if item not in id_to_index]
    if missing:
        raise FileNotFoundError(
            f"Missing {len(missing)} exp5 pose sequences for the exact split. "
            "Extract the full curated set in notebook 02."
        )
    train_ids, test_ids = exp5_exact_split(EXP5_ORDER)
    train_index = np.asarray([id_to_index[item] for item in train_ids])
    test_index = np.asarray([id_to_index[item] for item in test_ids])
    if MODE == "real":
        print("exp5 test composition (21 rows):")
        display(pd.Series(labels[test_index]).value_counts().reindex(CLASS_ORDER))
        print("exp5 train composition (47 rows):")
        display(pd.Series(labels[train_index]).value_counts().reindex(CLASS_ORDER))
        split_name = "exp5_exact_video_confounded"
        assert len(train_index) == 47 and len(test_index) == 21
else:
    from sklearn.model_selection import train_test_split
    all_index = np.arange(len(labels))
    train_index, test_index = train_test_split(
        all_index,
        train_size=0.70,
        random_state=42,
        stratify=labels,
    )
    train_ids = sequence_ids[train_index].tolist()
    test_ids = sequence_ids[test_index].tolist()
    split_name = "smoke_stratified_not_exp5"

split_rows = []
split_predictions = {}
for name in SAMPLER_NAMES:
    for seed in SEEDS:
        features = embeddings_by_arm[(name, seed)]
        classifier = make_rf()
        classifier.fit(features[train_index], labels[train_index])
        prediction = classifier.predict(features[test_index])
        metrics = score_predictions(labels[test_index], prediction)
        split_predictions[(name, seed)] = prediction
        split_rows.append({
            "mode": MODE,
            "mask": name,
            "seed": seed,
            "split": split_name,
            "train_sequences": len(train_index),
            "test_sequences": len(test_index),
            **metrics,
        })
split_metrics = pd.DataFrame(split_rows)
split_metrics.to_csv(
    ARTIFACT_DIR / "09_ablation_split_metrics.csv", index=False
)
display(split_metrics)
pretraining_ids = set(normal_ids.tolist())
overlap = len(set(sequence_ids[test_index].tolist()) & pretraining_ids)
print("split lane:", split_name)
print("test sequences that were seen during self-supervised pretraining:",
      overlap)

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut

video_census = (
    pd.DataFrame({"condition": labels, "video_id": video_ids})
    .groupby("condition")["video_id"]
    .nunique()
    .reindex(CONDITIONS)
)
display(video_census.rename("unique_videos").to_frame())
grouped_metrics = None
if int(video_census.min()) < 2:
    print(
        "BLOCKED: five-class video-grouped evaluation needs at least two "
        "source videos in every class. Real mode has one normal video, so a "
        "grouped fold would remove the entire normal class from training or "
        "place every normal sequence in one test fold. No grouped score is "
        "reported; see the census above."
    )
else:
    logo = LeaveOneGroupOut()
    grouped_rows = []
    for name in SAMPLER_NAMES:
        for seed in SEEDS:
            features = embeddings_by_arm[(name, seed)]
            fold_scores = []
            for train_pos, test_pos in logo.split(features, labels, video_ids):
                if not set(CONDITIONS).issubset(set(labels[train_pos])):
                    continue
                classifier = make_rf()
                classifier.fit(features[train_pos], labels[train_pos])
                prediction = classifier.predict(features[test_pos])
                fold_scores.append(
                    balanced_accuracy_score(labels[test_pos], prediction)
                )
            if not fold_scores:
                raise RuntimeError("No valid video-grouped fold was found")
            grouped_rows.append({
                "mode": MODE,
                "mask": name,
                "seed": seed,
                "grouped_folds": len(fold_scores),
                "grouped_balanced_accuracy": float(np.mean(fold_scores)),
            })
    grouped_metrics = pd.DataFrame(grouped_rows)
    grouped_metrics.to_csv(
        ARTIFACT_DIR / "09_ablation_grouped_metrics.csv", index=False
    )
    display(grouped_metrics)

In [ ]:
from scipy.signal import find_peaks
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import hashlib


def cadence_proxy(sequence):
    """Minimal stride-frequency proxy: ankle horizontal-x cycles per frame."""
    total = 0
    for joint in (27, 28):
        signal = np.asarray(sequence[:, joint, 0], dtype=np.float32)
        total += len(find_peaks(signal, distance=3)[0])
    return total / (2.0 * len(sequence))


cadences = np.asarray(
    [cadence_proxy(seq) for seq in all_xyz], dtype=np.float32
)
if MODE == "smoke":
    jitter = np.asarray([
        int(hashlib.sha1(item.encode("utf-8")).hexdigest()[:8], 16) % 1000
        / 1000.0
        for item in sequence_ids
    ], dtype=np.float32)
    cadences = cadences + 0.002 * (jitter - 0.5)
    print("smoke fixture: deterministic cadence jitter added for target variance")
print(
    "Caveat: this target is a cycles-per-frame proxy on the "
    "time-normalized 64-frame clips, not real Hz cadence. Absolute "
    "cadence from raw fps is notebook 08's question, so treat the "
    "near-zero R2 here as a weak discriminator, not a verdict."
)
print("cadence proxy range:", float(cadences.min()), float(cadences.max()))


def ridge_cv_r2(features, target):
    scores = []
    splitter = KFold(n_splits=5, shuffle=True, random_state=42)
    for train_pos, test_pos in splitter.split(features):
        probe = RidgeCV(alphas=np.logspace(-4, 2, 13))
        probe.fit(features[train_pos], target[train_pos])
        scores.append(
            r2_score(target[test_pos], probe.predict(features[test_pos]))
        )
    return float(np.mean(scores))


def coordinate_stat_features(xyz, valid):
    """12-d raw-coordinate baseline: mean and std over both joint regions."""
    rows = []
    for seq, ok in zip(xyz, valid):
        features = []
        for joints in (np.arange(33), np.asarray(MASK_KEYPOINTS, dtype=int)):
            arr = np.asarray(seq[:, joints], dtype=np.float64)
            validity = np.asarray(ok[:, joints], dtype=bool)
            for stat in ("mean", "std"):
                for coordinate in range(3):
                    values = arr[..., coordinate][validity]
                    if values.size == 0:
                        features.append(0.0)
                    elif stat == "mean":
                        features.append(float(values.mean()))
                    else:
                        features.append(float(values.std()))
        rows.append(features)
    return np.asarray(rows, dtype=np.float32)


coordinate_baseline = coordinate_stat_features(all_xyz, all_valid)
missingness_baseline = np.concatenate(
    [all_valid.mean(axis=1), all_valid.mean(axis=2)], axis=1
).astype(np.float32)
raw_r2 = ridge_cv_r2(coordinate_baseline, cadences)
missing_r2 = ridge_cv_r2(missingness_baseline, cadences)
print("raw-coordinate cadence R2:", raw_r2)
print("missingness cadence R2:", missing_r2)

cadence_rows = []
for name in SAMPLER_NAMES:
    for seed in SEEDS:
        latent_r2 = ridge_cv_r2(embeddings_by_arm[(name, seed)], cadences)
        cadence_rows.append({
            "mode": MODE,
            "mask": name,
            "seed": seed,
            "cadence_r2": latent_r2,
            "cadence_raw_coordinate_r2": raw_r2,
            "cadence_missingness_r2": missing_r2,
        })
cadence_metrics = pd.DataFrame(cadence_rows)
cadence_metrics.to_csv(
    ARTIFACT_DIR / "09_ablation_cadence.csv", index=False
)
display(cadence_metrics)

## Reading the comparison table and figure

The final table has one row per mask arm with the exp5-style lane scores, the grouped lane score when it is not blocked, the collapse statistics, and the cadence R2. Smoke mode aggregates the mean and standard deviation over its three seeds; real mode reports its single seed. The figure has three panels: lane accuracy, collapse diagnostics, and cadence R2 against the two baselines.

Read the table relatively. The arms share one small normal corpus from a single source video, and every normal test row was seen during self-supervised pretraining, so absolute classifier accuracy is transductive and inflated. What the ablation can still answer is whether one mask geometry beats another under identical conditions. If two arms sit within the seed spread on every metric, that is itself a finding about how much geometry matters.

In [ ]:
import matplotlib.pyplot as plt

detail_rows = []
for name in SAMPLER_NAMES:
    for seed in SEEDS:
        split_row = split_metrics.loc[
            (split_metrics["mask"] == name) & (split_metrics["seed"] == seed)
        ].iloc[0]
        cadence_row = cadence_metrics.loc[
            (cadence_metrics["mask"] == name) & (cadence_metrics["seed"] == seed)
        ].iloc[0]
        history_last = all_history.loc[
            (all_history["mask"] == name)
            & (all_history["seed"] == seed)
            & (all_history["epoch"] == EPOCHS)
        ].sort_values("epoch")
        last_row = history_last.iloc[-1]
        grouped_value = float("nan")
        if grouped_metrics is not None and len(grouped_metrics):
            grouped_hit = grouped_metrics.loc[
                (grouped_metrics["mask"] == name)
                & (grouped_metrics["seed"] == seed)
            ]
            if len(grouped_hit):
                grouped_value = float(
                    grouped_hit["grouped_balanced_accuracy"].iloc[0]
                )
        detail_rows.append({
            "mode": MODE,
            "mask": name,
            "seed": seed,
            "accuracy": float(split_row["accuracy"]),
            "balanced_accuracy": float(split_row["balanced_accuracy"]),
            "macro_f1": float(split_row["macro_f1"]),
            "grouped_balanced_accuracy": grouped_value,
            "feature_std": float(last_row["feature_std"]),
            "mean_pair_cosine": float(last_row["mean_pair_cosine"]),
            "final_loss": float(last_row["loss"]),
            "cadence_r2": float(cadence_row["cadence_r2"]),
        })
detail_metrics = pd.DataFrame(detail_rows)
detail_metrics.to_csv(ARTIFACT_DIR / "09_ablation_metrics.csv", index=False)

SUMMARY_COLUMNS = [
    "accuracy", "balanced_accuracy", "macro_f1",
    "grouped_balanced_accuracy", "feature_std", "mean_pair_cosine",
    "final_loss", "cadence_r2",
]


def summarize(series):
    numeric = pd.to_numeric(series, errors="coerce").dropna()
    if numeric.empty:
        return float("nan")
    return float(numeric.mean())


def summarize_std(series):
    numeric = pd.to_numeric(series, errors="coerce").dropna()
    if len(numeric) < 2:
        return float("nan")
    return float(numeric.std())


summary_rows = []
for name in SAMPLER_NAMES:
    arm_rows = detail_metrics[detail_metrics["mask"] == name]
    row = {"mask": name, "runs": int(len(arm_rows))}
    for column in SUMMARY_COLUMNS:
        row[f"{column}_mean"] = summarize(arm_rows[column])
        row[f"{column}_std"] = summarize_std(arm_rows[column])
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
summary.to_csv(ARTIFACT_DIR / "09_ablation_summary.csv", index=False)
display(summary)
if len(SEEDS) < 2:
    print(
        "Single-seed run: standard-deviation columns are NaN, not measured "
        "zero variance. Repeat with several seeds before quoting spreads."
    )

names = list(summary["mask"])
positions = np.arange(len(names))


def series_pair(column):
    return (
        [summary.loc[index, f"{column}_mean"] for index in range(len(names))],
        [summary.loc[index, f"{column}_std"] for index in range(len(names))],
    )


balanced_mean, balanced_std = series_pair("accuracy")
feature_mean, feature_std = series_pair("feature_std")
cosine_mean, cosine_std = series_pair("mean_pair_cosine")
cadence_mean, cadence_std = series_pair("cadence_r2")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
width = 0.34
axes[0].bar(
    positions, balanced_mean, yerr=balanced_std, capsize=3, width=width,
    color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"],
)
axes[0].set_xticks(positions)
axes[0].set_xticklabels(names, rotation=20, ha="right")
axes[0].set_title("exp5-style lane accuracy (single seed where std is NaN)")
axes[0].set_ylabel("accuracy")

axes[1].bar(
    positions - width / 2, feature_mean, yerr=feature_std, capsize=3,
    width=width, label="feature std (higher is healthier)",
)
axes[1].bar(
    positions + width / 2, cosine_mean, yerr=cosine_std, capsize=3,
    width=width, label="pair cosine (lower is healthier)",
)
axes[1].set_xticks(positions)
axes[1].set_xticklabels(names, rotation=20, ha="right")
axes[1].set_title("collapse diagnostics on the normal corpus")
axes[1].legend(fontsize=8)

axes[2].bar(
    positions, cadence_mean, yerr=cadence_std, capsize=3, width=width,
    color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"],
    label="latent Ridge R2",
)
axes[2].axhline(raw_r2, color="black", linestyle="--", label="raw coordinates")
axes[2].axhline(missing_r2, color="grey", linestyle=":", label="missingness")
axes[2].set_xticks(positions)
axes[2].set_xticklabels(names, rotation=20, ha="right")
axes[2].set_title("minimal cadence probe (Ridge R2)")
axes[2].legend(fontsize=8)

fig.tight_layout()
fig.savefig(
    ARTIFACT_DIR / "09_ablation_comparison_figure.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

if MODE == "real" and len(SEEDS) == 1:
    print(
        "Prediction-agreement check across arms (same 21 test rows): "
        "pairs can produce identical label vectors even though the four "
        "checkpoints differ, because a depth-5 Random Forest on 21 rows "
        "has coarse decision boundaries."
    )
    agreements = []
    for a_name in SAMPLER_NAMES:
        for b_name in SAMPLER_NAMES:
            if a_name >= b_name:
                continue
            a_pred = split_predictions.get((a_name, SEEDS[0]))
            b_pred = split_predictions.get((b_name, SEEDS[0]))
            if a_pred is not None and b_pred is not None:
                agreement = float(np.mean(a_pred == b_pred))
                agreements.append((a_name, b_name, agreement))
    if agreements:
        for row in agreements:
            print(f"  {row[0]} vs {row[1]}: label agreement {row[2]:.3f}")

best_classifier = summary.loc[
    summary["accuracy_mean"].idxmax()
]
best_cadence = summary.loc[summary["cadence_r2_mean"].idxmax()]
print("best exp5-lane accuracy:", best_classifier["mask"],
      f"{float(best_classifier['accuracy_mean']):.3f}")
print("best cadence probe:", best_cadence["mask"],
      f"{float(best_cadence['cadence_r2_mean']):.3f}")
print("cadence baselines - raw coordinates:", f"{raw_r2:.3f}",
      "missingness:", f"{missing_r2:.3f}")
print("Read rows relative to the seed spread; smoke rows are fixtures.")

## Verdict: does the literature-guided mask help?

Ask four questions of the table:

1. Does neurologic-10 beat random-10 and full-body-33 on the exp5-style lane? If yes, restricting targets to the mapped joints earns its keep. If the arms tie, the joint restriction is not hurting, but it is also not the source of the score.
2. Does motion-aware-10 change anything over neurologic-10? The paper used motion-aware masking and this project deliberately replaced it with uniform sampling, so a large gap would reopen that design discussion, while a tie would support the uniform choice.
3. Do collapse diagnostics differ across arms? A geometry that collapses the representation is disqualifying no matter what the classifier says.
4. Is cadence more linearly decodable from one arm's latent than from the raw-coordinate baseline?

Two boundaries stay fixed regardless of the verdict. Motion-aware masking is evaluated only inside the allowed neurologic set, so no forbidden joint is ever hidden, which keeps checklist rule 10 intact. And every conclusion here is relative and transductive; a real claim about clinical gait needs independent normal videos, which the census check in the grouped lane refuses to fake.